# **1. 단순 선형 회귀 분석**
- 전복의 나이를 예측하는 선형회귀모델을 생성하세요.
- 전복의 ‘성별’, ‘키’, ‘지름’, ‘높이’, ‘전체무게’, ‘몸통무게’, ‘내장무게’, ‘껍질무게’를 이용해 ‘껍질의 고리 수’를 예측한 뒤, **예측된 ‘껍질의 고리 수’에 1.5를 더하면 전복의 나이**가 됩니다.

In [1]:
# 기본 모듈 불러오기
import numpy as np
import pandas as pd

**1) 데이터 load 및 변형**

In [7]:
from google.colab import files
uploaded = files.upload()

Saving abalone.csv to abalone.csv


In [8]:
# 데이터 로드
data = pd.read_csv("abalone.csv")
data.head()
print(data.shape)

# 성별 M은 Male, F는 Female, I는 Infant 이므로 따로 열 만들기
for label in "MFI":
    data[label] = data["Sex"] == label
data.drop('Sex', axis=1, inplace=True)

(4177, 9)


**2) X, y 선택**
: y는 Rings열, X는 Rings열을 제외한 나머지를 선택하되 전부 실수가 되도록 한다.

In [9]:
# X,y 데이터 선택
y = data.Rings.values
data.drop('Rings', axis=1, inplace=True)

X =  data.values.astype(float)

 **3) train/test set 분리**

In [19]:
# 필요한 모듈 불러오기
X = data.values.astype(float)
from sklearn.model_selection import train_test_split

In [20]:
# train과 test set 분리 (train:test = 7:3 비율로)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)

**4) 선형회귀모델 생성, 모델 예측치 구하기**

In [21]:
#필요한 모듈 불러오기
from sklearn.linear_model import LinearRegression

In [22]:
#선형회귀모델 생성 및 훈련
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

LinearRegression()

In [23]:
# 모델 예측치 구하기
lr_model.predict(X_test)
# 모델 예측치를 활용해 최종적으로 전복의 나이를 예측
lr_model.predict(X_test) + 1.5

array([14.72319148, 10.63647192, 11.82013209, ..., 10.7343318 ,
       20.24528598, 12.46661461])

**5) 모델 평가: MSE, RMSE, R2 score, corr 구하기**

In [24]:
#필요한 모듈 불러오기
from sklearn.metrics import mean_squared_error, r2_score

- MSE, RMSE

In [25]:
#mse, rmse
mse = mean_squared_error(y_test, lr_model.predict(X_test))
rmse = np.sqrt(mse)
print(mse, rmse)

5.003851007700237 2.236928923256221


- R2 score

In [26]:
#R2 score 측정
R2 = r2_score(y_test, lr_model.predict(X_test))
print(R2)

0.5253868694528043


- 회귀 절편값

In [27]:
#회귀 절편 값
print('절편 값:', lr_model.intercept_)

절편 값: 3.7449653614603235


- 회귀 계수 값

In [28]:
#회귀 계수 값
co_df = pd.DataFrame(lr_model.coef_, index=data.columns, columns=['회귀 계수'])
co_df

,회귀 계수
Length,-0.090922
Diameter,11.173833
Height,7.133454
Whole weight,9.021445
Shucked weight,-20.005803
Viscera weight,-10.693361
Shell weight,9.612902
M,0.326268
F,0.228953
I,-0.555221


- 상관계수

Hint: corr 함수 이용.

In [29]:
# 상관계수 구하기
corelation = data.corr()
corelation

,Length,Diameter,Height,Whole weight,Shucked weight,Viscera weight,Shell weight,M,F,I
Length,1.000000,0.986812,0.827554,0.925261,0.897914,0.903018,0.897706,0.236543,0.309666,-0.551465
Diameter,0.986812,1.000000,0.833684,0.925452,0.893162,0.899724,0.905330,0.240376,0.318626,-0.564315
Height,0.827554,0.833684,1.000000,0.819221,0.774972,0.798319,0.817338,0.215459,0.298421,-0.518552
Whole weight,0.925261,0.925452,0.819221,1.000000,0.969405,0.966375,0.955355,0.252038,0.299741,-0.557592
Shucked weight,0.897914,0.893162,0.774972,0.969405,1.000000,0.931961,0.882617,0.251793,0.263991,-0.521842
Viscera weight,0.903018,0.899724,0.798319,0.966375,0.931961,1.000000,0.907656,0.242194,0.308444,-0.556081
Shell weight,0.897706,0.905330,0.817338,0.955355,0.882617,0.907656,1.000000,0.235391,0.306319,-0.546953
M,0.236543,0.240376,0.215459,0.252038,0.251793,0.242194,0.235391,1.000000,-0.512528,-0.522541
F,0.309666,0.318626,0.298421,0.299741,0.263991,0.308444,0.306319,-0.512528,1.000000,-0.464298
I,-0.551465,-0.564315,-0.518552,-0.557592,-0.521842,-0.556081,-0.546953,-0.522541,-0.464298,1.000000


# **2. Polynomial features**

In [15]:
# PolynomialFeatures 라이브러리 호출
from sklearn.preprocessing import PolynomialFeatures

In [16]:
# 임의 데이터 생성

X = np.arange(6).reshape(3, 2)

df =  pd.DataFrame(X)
df.columns = ['x_1','x_2']
df

,x_1,x_2
0,0,1
1,2,3
2,4,5


In [17]:
# 차원은 2로 설정
# fit_transform 메소드를 통해 데이터 변환
# PolynomialFeatures로 변환 된 데이터를 데이터 프레임 형태로 변환
poly = PolynomialFeatures(degree=2)
poly_ftr = poly.fit_transform(X)
df_poly = pd.DataFrame(poly_ftr)

In [18]:
# df_poly의 컬럼을 1,x1,x2,x1^2,x1*x2,x2^2 로 변경
df_poly.columns = ['1','x1','x2','x1^2','x1*x2','x2^2']
df_poly

,1,x1,x2,x1^2,x1*x2,x2^2
0,1.0,0.0,1.0,0.0,0.0,1.0
1,1.0,2.0,3.0,4.0,6.0,9.0
2,1.0,4.0,5.0,16.0,20.0,25.0
